# Normative Modeling Walkthrough with PCNtoolkit

**Normative modeling** characterizes variation across a healthy population and then measures how much an individual *deviates* from that norm. Instead of asking "is this patient in group A or B?", it asks "how unusual is this person's brain, relative to what is expected for their age and sex?"

This notebook walks through a complete pipeline:
1. Generate synthetic neuroimaging-like data
2. Fit a Bayesian Linear Regression (BLR) normative model
3. Evaluate performance metrics
4. Visualise centile curves and Z-score distributions

## 1. Imports & Setup

In [ ]:
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from scipy.stats import norm  # moved here from the visualisation cell

import pcntoolkit
import pcntoolkit.normative as normative

# Reproducibility
np.random.seed(42)

# Create output directories
os.makedirs('../data', exist_ok=True)
os.makedirs('../outputs/blr', exist_ok=True)

print('PCNtoolkit version:', pcntoolkit.__version__)
print('Directories ready.')


## 2. Synthetic Data Generation

We simulate a brain feature (e.g. mean cortical thickness in mm) that shows a realistic age-related decline, with added Gaussian noise.

In [ ]:
N_TRAIN = 500
N_TEST  = 100
N_TOTAL = N_TRAIN + N_TEST

# Covariates: age (18-80) and sex (0=female, 1=male)
age = np.random.uniform(18, 80, N_TOTAL)
sex = np.random.randint(0, 2, N_TOTAL).astype(float)

# Brain feature: linear age decline + sex offset + noise
# Cortical thickness ~ 2.8 - 0.005 * age - 0.05 * sex + noise
noise = np.random.normal(0, 0.1, N_TOTAL)
ct    = 2.8 - 0.005 * age - 0.05 * sex + noise

df = pd.DataFrame({'age': age, 'sex': sex, 'cortical_thickness': ct})
df.describe().round(3)

## 3. Data Preparation

PCNtoolkit expects plain text files: covariates and responses as space-delimited `.txt` tables.

In [ ]:
train = df.iloc[:N_TRAIN]
test  = df.iloc[N_TRAIN:]

def save_txt(arr, path):
    """Save a numpy array as a space-delimited text file."""
    np.savetxt(path, arr, fmt='%.6f')

# Covariates: [age, sex] — shape (N, 2)
save_txt(train[['age', 'sex']].values,   '../data/cov_train.txt')
save_txt(test[['age', 'sex']].values,    '../data/cov_test.txt')

# Response: [cortical_thickness] — shape (N, 1)
save_txt(train[['cortical_thickness']].values, '../data/resp_train.txt')
save_txt(test[['cortical_thickness']].values,  '../data/resp_test.txt')

print('Saved train/test files to ../data/')

## 4. Model Fitting

We call `pcntoolkit.normative.estimate()` with a Bayesian Linear Regression (BLR) kernel.
The function trains on the training set and generates predictions on the test set.

In [ ]:
normative.estimate(
    covfile='../data/cov_train.txt',
    respfile='../data/resp_train.txt',
    testcov='../data/cov_test.txt',
    testresp='../data/resp_test.txt',
    alg='blr',
    outputsuffix='_blr',
    savemodel=True,
    output_path='../outputs/blr/'
)

print('Model fitting complete. Outputs saved to ../outputs/blr/')

## 5. Evaluation

Load the output files and inspect the performance metrics.

In [ ]:
out = '../outputs/blr/'

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Predictions and uncertainties
yhat = load_pkl(os.path.join(out, 'yhat_blr.pkl')).flatten()   # predicted mean
ys2  = load_pkl(os.path.join(out, 'ys2_blr.pkl')).flatten()    # predicted variance
Z    = load_pkl(os.path.join(out, 'Z_blr.pkl')).flatten()       # deviation Z-scores

# Summary metrics (text files contain one value per brain feature)
def read_metric(path):
    val = np.loadtxt(path)
    return float(np.atleast_1d(val)[0])  # safe for both scalar and array outputs

metrics = {
    'MSLL' : read_metric(os.path.join(out, 'MSLL_blr.txt')),
    'EXPV' : read_metric(os.path.join(out, 'EXPV_blr.txt')),
    'SMSE' : read_metric(os.path.join(out, 'SMSE_blr.txt')),
    'RMSE' : read_metric(os.path.join(out, 'RMSE_blr.txt')),
}

print('\nPerformance metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

print(f'\nZ-score stats: mean={Z.mean():.3f}, std={Z.std():.3f}')


## 6. Visualisation

### 6a. Centile curves — predicted distribution vs age

In [ ]:
age_test = test['age'].values
ct_test  = test['cortical_thickness'].values
sigma    = np.sqrt(np.clip(ys2, 0, None))  # clip negatives from numerical noise

sort_idx = np.argsort(age_test)
age_s    = age_test[sort_idx]
yhat_s   = yhat[sort_idx]
sigma_s  = sigma[sort_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(age_s, yhat_s - 2*sigma_s, yhat_s + 2*sigma_s,
                alpha=0.15, color='steelblue', label='±2 SD (≈95%)')
ax.fill_between(age_s, yhat_s - sigma_s,   yhat_s + sigma_s,
                alpha=0.3,  color='steelblue', label='±1 SD (≈68%)')
ax.plot(age_s, yhat_s, color='steelblue', linewidth=2, label='Predicted mean')
ax.scatter(age_test, ct_test, s=15, alpha=0.6, color='black', label='Observed (test)')

ax.set_xlabel('Age (years)')
ax.set_ylabel('Cortical Thickness (mm)')
ax.set_title('Normative Centile Curves')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/blr/centile_curves.png', dpi=150)
plt.show()


### 6b. Z-score distribution vs standard normal

In [ ]:
frac_extreme = np.mean(np.abs(Z) > 1.96) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(Z, bins=20, density=True, alpha=0.6, color='steelblue', label='Observed Z')

x = np.linspace(-4, 4, 200)
ax.plot(x, norm.pdf(x), 'r--', linewidth=2, label='N(0,1) reference')
ax.axvline(-1.96, color='grey', linestyle=':', linewidth=1)
ax.axvline( 1.96, color='grey', linestyle=':', linewidth=1, label='±1.96 threshold')

ax.set_xlabel('Z-score')
ax.set_ylabel('Density')
ax.set_title(f'Z-score Distribution  ({frac_extreme:.1f}% outside ±1.96)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../outputs/blr/z_score_distribution.png', dpi=150)
plt.show()
